In [0]:
%pip install --upgrade databricks-sdk pyyaml jsonschema
dbutils.library.restartPython()

# Init Ontology Project

Validates the Genie Ontology Modeled Knowledge SDK inputs (`sdk_config.json` contract), checks Unity Catalog access, then scaffolds a run project (both bundles, run kit, docs folders) with `genie_ontology_modeled_knowledge_sdk.scaffold.scaffold_project`.

* The SDK is used as an installed package: set `sdk_wheel` to the published wheel (`/Volumes/<sdk_catalog>/<sdk_schema>/sdk_wheels/.../genie_ontology_modeled_knowledge_sdk-<version>-py3-none-any.whl`, see `docs/distribution/README.md`). Only when the package is not installed and this notebook sits in an SDK checkout does it fall back to `<sdk_root>/src` (developer mode; it prints that it did).
* `vendor` = `true` copies the SDK modules the project notebooks import into the project (opt-in, for workspaces without package egress). The default installs the wheel instead.

* Run `notebooks/setup_sdk` once per workspace first (SDK catalog and schema, SDK bundle deploy, skills publishing).
* Required widgets start empty and the notebook does nothing until they are set. The `example` dropdown (`none`, `tpch`) prefills the inputs from `examples/tpch/sdk_config.json`.
* After scaffolding it checks the prerequisites (catalog and CREATE SCHEMA, warehouse, Git folder on a feature branch, skills, Pages domain, SDK wheel), writes both gitignored `variable-overrides.json` files from the same inputs, writes the genie-build-waves vars file when `vars_file` is set (outside the project), and prints a checklist with the exact deploy, setup and run steps.
* List widgets take comma-separated values.
* Leave `target_dir` empty to run only the validation and access check.
* Existing files are never overwritten unless `force` is `true`. The scaffold never touches `.databricks/`; the setup step writes only each bundle's `.databricks/bundle/<target>/variable-overrides.json` and keeps an edited one unless `force` is `true`.

In [0]:
# One widget per sdk_config.json field (schemas/sdk_config.schema.json), plus scaffold and setup options.
# Required inputs have empty defaults: the notebook does nothing until they are set. The example
# dropdown loads examples/tpch/sdk_config.json from the SDK; non-empty widgets override its values.
dbutils.widgets.dropdown("example", "none", ["none", "tpch"], "example (prefill sdk_config)")
dbutils.widgets.text("domain_name", "", "domain_name")
dbutils.widgets.text("domain_context", "", "domain_context")
dbutils.widgets.text("source_catalogs", "", "source_catalogs (csv)")
dbutils.widgets.text("source_schemas", "", "source_schemas (csv, catalog.schema)")
dbutils.widgets.text("source_tables", "", "source_tables (csv, catalog.schema.table)")
dbutils.widgets.text("target_catalog", "", "target_catalog")
dbutils.widgets.text("target_schema", "", "target_schema")
dbutils.widgets.text("supplemental_volume_dirs", "", "supplemental_volume_dirs (csv)")
dbutils.widgets.text("dashboard_inputs", "", "dashboard_inputs (csv)")
dbutils.widgets.text("source_catalog_for_reads", "", "source_catalog_for_reads")
dbutils.widgets.text("source_schema_for_reads", "", "source_schema_for_reads")
dbutils.widgets.text("warehouse_id", "", "warehouse_id")
dbutils.widgets.text("max_metric_views_per_agent", "50", "max_metric_views_per_agent")
dbutils.widgets.text("repo_path", "", "repo_path")
# Scaffold options
dbutils.widgets.text("target_dir", "", "target_dir (empty = check only)")
dbutils.widgets.text("project_name", "", "project_name (default: target_dir name)")
dbutils.widgets.text("bundle_target", "dev", "bundle_target")
dbutils.widgets.dropdown("force", "false", ["false", "true"], "force overwrite")
dbutils.widgets.text("sdk_root", "", "sdk_root (templates; default: this notebook's repo)")
dbutils.widgets.dropdown("vendor", "false", ["false", "true"], "vendor SDK modules (opt-in)")
dbutils.widgets.text("sdk_wheel", "", "sdk_wheel (/Volumes/... .whl; empty = already installed or SDK checkout)")
# Project setup: both variable-overrides.json files, prerequisite checks, genie-build-waves vars file
dbutils.widgets.text("consumer_group", "", "consumer_group (reads metric views, runs agents)")
dbutils.widgets.text("curator_group", "", "curator_group (edits agents)")
dbutils.widgets.text("sdk_volume_path", "", "sdk_volume_path (/Volumes/<sdk_catalog>/<sdk_schema>/sdk_wheels)")
dbutils.widgets.text("sdk_version", "", "sdk_version (default: installed SDK version)")
dbutils.widgets.text("skills_catalog", "", "skills_catalog (SDK catalog)")
dbutils.widgets.text("skills_schema", "", "skills_schema (resolved SDK schema)")
dbutils.widgets.text("pages_domain_id", "", "pages_domain_id (optional Discover domain)")
dbutils.widgets.text("branch", "", "branch (feature branch of the project Git folder)")
dbutils.widgets.text("vars_file", "", "vars_file (outside the project; empty = skip)")
dbutils.widgets.text("genie_build_waves_dir", "", "genie_build_waves_dir (checkout path)")
dbutils.widgets.text("instance_suffix", "", "instance_suffix (default: -<project_name>)")

In [0]:
# Installs the published SDK wheel named by the sdk_wheel widget, then restarts Python
# (widgets keep their values). Empty: use an already installed package, or the developer
# fallback in the next cell when this notebook sits in an SDK checkout.
_sdk_wheel = dbutils.widgets.get("sdk_wheel").strip()
if _sdk_wheel:
    get_ipython().run_line_magic("pip", f"install --upgrade databricks-sdk {_sdk_wheel}")
    dbutils.library.restartPython()
else:
    print("sdk_wheel is empty: using the installed package or the SDK checkout.")

In [0]:
import json
import sys
from pathlib import Path


def _w(name: str) -> str:
    return dbutils.widgets.get(name).strip()


def _csv(name: str) -> list[str]:
    return [v.strip() for v in _w(name).split(",") if v.strip()]


# SDK root: widget, else two levels above this notebook (<sdk>/notebooks/init_ontology_project).
if _w("sdk_root"):
    sdk_root = Path(_w("sdk_root"))
else:
    nb_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
    sdk_root = Path("/Workspace" + nb_path).parent.parent
# Installed package first (wheel). Developer fallback only when it is missing AND this
# notebook runs from an SDK checkout: add <sdk_root>/src and say so.
try:
    import genie_ontology_modeled_knowledge_sdk  # noqa: F401
except ImportError:
    if not (sdk_root / "src" / "genie_ontology_modeled_knowledge_sdk" / "__init__.py").is_file():
        raise ImportError(
            "genie_ontology_modeled_knowledge_sdk is not installed: set the sdk_wheel widget to the published wheel "
            "and re-run the install cell (see docs/distribution/README.md)."
        ) from None
    sys.path.insert(0, str(sdk_root / "src"))
    print(f"Developer mode: SDK package not installed, importing from the checkout at {sdk_root / 'src'}")

import genie_ontology_modeled_knowledge_sdk as _sdk  # noqa: E402
from genie_ontology_modeled_knowledge_sdk.config import check_access, config_from_dict, validate_config  # noqa: E402
from genie_ontology_modeled_knowledge_sdk import setup_project as sp  # noqa: E402
from genie_ontology_modeled_knowledge_sdk.scaffold import scaffold_project  # noqa: E402

raw = {
    "domain_name": _w("domain_name"),
    "domain_context": _w("domain_context"),
    "source_catalogs": _csv("source_catalogs"),
    "source_schemas": _csv("source_schemas"),
    "source_tables": _csv("source_tables"),
    "target_catalog": _w("target_catalog"),
    "target_schema": _w("target_schema"),
    "supplemental_volume_dirs": _csv("supplemental_volume_dirs"),
    "dashboard_inputs": _csv("dashboard_inputs"),
    "source_catalog_for_reads": _w("source_catalog_for_reads"),
    "source_schema_for_reads": _w("source_schema_for_reads"),
    "warehouse_id": _w("warehouse_id"),
    "repo_path": _w("repo_path"),
}
if _w("max_metric_views_per_agent"):
    raw["max_metric_views_per_agent"] = int(_w("max_metric_views_per_agent"))
# Drop empty optional values so the dict matches the schema (no empty strings / lists).
raw = {k: v for k, v in raw.items() if v not in ("", [], None)}
# Example values (examples/tpch/sdk_config.json in the SDK) fill only the widgets left empty.
if _w("example") == "tpch":
    example = json.loads((sdk_root / "examples/tpch/sdk_config.json").read_text())
    raw = {**example, **raw}
    print("Loaded example values from examples/tpch/sdk_config.json (non-empty widgets win).")

# Do nothing until the required inputs are set.
_has_source = any(raw.get(k) for k in ("source_catalogs", "source_schemas", "source_tables"))
missing_inputs = [k for k in ("domain_name", "domain_context", "target_catalog", "target_schema") if not raw.get(k)]
missing_inputs += [] if _has_source else ["source_tables (or source_schemas / source_catalogs)"]
if missing_inputs:
    print("Set the required widgets (or choose an example), then re-run: " + ", ".join(missing_inputs))
    dbutils.notebook.exit(json.dumps({"ok": False, "missing": missing_inputs}))
print(f"SDK {_sdk.__version__} from {Path(_sdk.__file__).parent}; templates from {sdk_root}")
print(json.dumps(raw, indent=2))

In [0]:
from databricks.sdk import WorkspaceClient

config_errors = validate_config(raw)
if config_errors:
    raise ValueError("sdk_config is invalid:\n  " + "\n  ".join(config_errors))
cfg = config_from_dict(raw)

access_problems = check_access(cfg, WorkspaceClient())
print(f"validate_config: {len(config_errors)} errors")
print(f"check_access: {len(access_problems)} problems")
for p in access_problems:
    print(f"  - {p}")
if access_problems:
    raise RuntimeError("Fix the access problems above before scaffolding.")

# Project setup inputs: the same values feed the prerequisite checks, both overrides files and the vars file.
ws = WorkspaceClient()
_src_catalog, _src_schema = sp.derive_sources(cfg)
project = sp.ProjectInputs(
    project_dir=_w("target_dir"),
    catalog=cfg.target_catalog,
    schema_name=cfg.target_schema,
    warehouse_id=cfg.warehouse_id or "",
    consumer_group=_w("consumer_group"),
    curator_group=_w("curator_group"),
    source_catalog=_src_catalog,
    source_schema=_src_schema,
    bundle_target=_w("bundle_target") or "dev",
    user_name=ws.current_user.me().user_name,
    vendor=_w("vendor") == "true",
    sdk_volume_path=_w("sdk_volume_path"),
    sdk_version=_w("sdk_version") or _sdk.__version__,
    skills_catalog=_w("skills_catalog"),
    skills_schema=_w("skills_schema"),
    pages_domain_id=_w("pages_domain_id"),
    project_name=_w("project_name"),
    branch=_w("branch"),
    instance_suffix=_w("instance_suffix"),
    vars_file=_w("vars_file"),
    genie_build_waves_dir=_w("genie_build_waves_dir"),
)
prereqs = sp.check_prereqs(ws, project) if project.project_dir else []
for c in prereqs:
    print(f"[{c.status:>6}] {c.name}: {c.detail}")

In [0]:
report = None
if _w("target_dir"):
    report = scaffold_project(
        cfg,
        _w("target_dir"),
        sdk_root,
        force=_w("force") == "true",
        project_name=_w("project_name") or None,
        bundle_target=_w("bundle_target") or "dev",
        vendor=_w("vendor") == "true",
    )
else:
    print("target_dir is empty: validation and access check only, nothing scaffolded.")

# Both gitignored variable-overrides.json files from the same inputs (edited files kept unless force).
overrides, vars_result = [], None
missing_setup = [m for m in sp.missing_project_inputs(project) if m != "project_dir"]
if report is not None and not missing_setup:
    overrides = sp.write_overrides(project.project_dir, project, force=_w("force") == "true")
    for r in overrides:
        print(f"overrides {r.action}: {r.path}")
    if project.vars_file:
        vars_result = sp.write_vars_file(project.vars_file, project, force=_w("force") == "true")
        print(f"vars file {vars_result.action}: {vars_result.path}")
elif report is not None:
    print("Overrides not written; set: " + ", ".join(missing_setup))

In [0]:
summary = {
    "domain_name": cfg.domain_name,
    "source_inputs": len(cfg.source_catalogs) + len(cfg.source_schemas) + len(cfg.source_tables),
    "config_errors": len(config_errors),
    "access_problems": len(access_problems),
    "scaffold": None if report is None else {
        "target_dir": report.target_dir,
        "files_written": report.files_written,
        "unchanged": len(report.unchanged),
        "skipped": report.skipped,
        "overwritten": report.overwritten,
    },
}
summary["prereqs"] = [c.to_dict() for c in prereqs]
summary["overrides"] = [r.to_dict() for r in overrides]
print(json.dumps(summary, indent=2))
commands = sp.project_commands(project) if report is not None else []
print(sp.checklist({
    "missing": missing_setup if report is not None else ["target_dir"],
    "checks": prereqs,
    "scaffold": report,
    "overrides": overrides,
    "vars_file": vars_result,
    "commands": commands,
}))
dbutils.notebook.exit(json.dumps(summary))